# Busca de consonância e afins

Dada uma ementa, a busca híbrida combina o vetor denso e o BM25 por RRF. Ela roda separadamente em cada grupo para que as muitas teses de temas não tirem as súmulas do resultado.

| Seção | Conteúdo | Pode indicar consonância? |
|---|---|---|
| Súmulas | Súmulas vigentes | Sim |
| Precedentes com tese | Temas, IAC, PUIL etc. com tese firmada | Sim |
| Em andamento | Temas afetados, ainda sem tese | Não, são apenas afins |
| Inativos | Súmulas e temas cancelados ou superados | Não, aparecem como alerta |

A aderência mede a proximidade entre os textos, não a concordância jurídica. Uma ementa que contraria a súmula também fica próxima dela. Quem avalia a relação é o notebook 06, e a palavra final é de quem revisa.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm sem ipywidgets

from dataclasses import dataclass

import numpy as np
import ollama
import re
from fastembed import SparseTextEmbedding
from IPython.display import Markdown, display
from qdrant_client import QdrantClient, models

QDRANT_URL = "http://localhost:6333"
COLECAO = "stj_precedentes"
MODELO_DENSO = "bge-m3"
CANDIDATOS_POR_VETOR = 50

# Cosseno do bge-m3, calibrado com poucos exemplos (ver o notebook 07). Paráfrases corretas ficaram entre 0,58 e
# 0,92; distratores chegaram a 0,67; temas sem relação ficaram abaixo de 0,53.
ADERENCIA_FORTE = 0.65
ADERENCIA_MINIMA = 0.50

qdrant = QdrantClient(url=QDRANT_URL, timeout=60)
bm25 = SparseTextEmbedding("Qdrant/bm25", language="portuguese")

In [2]:
RE_INSTRUCAO = re.compile(r"^\s*(verifique|verificar|busque|procure|encontre|há|existe)\b.*?(abaixo|seguinte)\s*:?\s*", re.I | re.S)


def limpar_query(query: str) -> str:
    """Tira o pedido ("verifique se existe súmula...") para sobrar só a ementa, que é o que deve ser comparado."""
    return RE_INSTRUCAO.sub("", query, count=1).replace("[...]", " ").strip().strip('"“”').strip()


@dataclass
class Candidato:
    citacao: str
    enunciado: str
    situacao: str
    aderencia: float
    nivel: str
    url: str
    payload: dict


def _filtro(**condicoes) -> models.Filter:
    return models.Filter(
        must=[models.FieldCondition(key=k, match=models.MatchValue(value=v)) for k, v in condicoes.items()]
    )


def _nivel(aderencia: float) -> str:
    return "forte" if aderencia >= ADERENCIA_FORTE else "moderada"


def buscar(ementa: str, filtro: models.Filter, limite: int) -> list[Candidato]:
    denso = np.array(ollama.embed(model=MODELO_DENSO, input=[ementa]).embeddings[0])
    denso = denso / np.linalg.norm(denso)
    esparso = next(iter(bm25.query_embed(ementa)))
    pontos = qdrant.query_points(
        COLECAO,
        prefetch=[
            models.Prefetch(query=denso.tolist(), using="denso", limit=CANDIDATOS_POR_VETOR, filter=filtro),
            models.Prefetch(
                query=models.SparseVector(indices=esparso.indices.tolist(), values=esparso.values.tolist()),
                using="bm25",
                limit=CANDIDATOS_POR_VETOR,
                filter=filtro,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limite * 4,
        with_payload=True,
        with_vectors=["denso"],
    ).points

    candidatos = []
    for ponto in pontos:
        vetor = np.array(ponto.vector["denso"])
        aderencia = float(denso @ (vetor / np.linalg.norm(vetor)))
        if aderencia < ADERENCIA_MINIMA:
            continue
        p = ponto.payload
        candidatos.append(Candidato(p["citacao"], p["enunciado"], p["situacao"], aderencia, _nivel(aderencia), p["url"], p))
    return candidatos[:limite]


def consonancia(query: str, por_secao: int = 5) -> dict[str, list[Candidato]]:
    ementa = limpar_query(query)
    inativos = [c for c in buscar(ementa, _filtro(vigente=False), por_secao) if c.nivel == "forte"]
    return {
        "Súmulas": buscar(ementa, _filtro(fonte="sumula", vigente=True), por_secao),
        "Precedentes com tese": buscar(ementa, _filtro(fonte="precedente", vigente=True, tem_tese=True), por_secao),
        "Em andamento (sem tese firmada)": buscar(ementa, _filtro(fonte="precedente", vigente=True, tem_tese=False), 3),
        "Inativos (não usar como fundamento)": inativos,
    }


def mostrar(resultado: dict[str, list[Candidato]]) -> None:
    linhas = []
    for secao, candidatos in resultado.items():
        linhas.append(f"### {secao}\n")
        if not candidatos:
            linhas.append("_Nenhum candidato acima do limiar._\n")
        for c in candidatos:
            titulo = f"[{c.citacao}]({c.url})" if c.url else c.citacao
            linhas.append(f"- **{titulo}**: aderência {c.nivel} ({c.aderencia:.3f}), {c.situacao}. {c.enunciado[:400]}")
        linhas.append("")
    display(Markdown("\n".join(linhas)))

## Exemplo com uma ementa sobre prisão preventiva

In [3]:
QUERY_EXEMPLO = """Verifique se exista alguma sumula em consonancia com a ementa abaixo

PRISÃO PREVENTIVA. PORTE ILEGAL DE ARMA DE FOGO DE USO PERMITIDO, TRÁFICO
DE DROGAS E ASSOCIAÇÃO PARA O TRÁFICO. GARANTIA DA ORDEM PÚBLICA. CONVERSÃO EX
OFFICIO DA PRISÃO EM FLAGRANTE EM PREVENTIVA. IMPOSSIBILIDADE. NECESSIDADE DE
REQUERIMENTO PRÉVIO OU PELO MINISTÉRIO PÚBLICO OU PELO QUERELANTE, OU PELO
ASSISTENTE OU, POR FIM, MEDIANTE REPRESENTAÇÃO DA AUTORIDADE POLICIAL. [...] No
caso, a decisão agravada deve ser mantida, uma vez que não é possível a
decretação da prisão preventiva de ofício em face do que dispõe a Lei n.
13.964/2019, mesmo se decorrente de prisão em flagrante e se não tiver ocorrido
audiência de custódia. Isso porque não existe diferença entre a conversão da
prisão em flagrante em preventiva e a decretação da prisão preventiva como uma
primeira prisão (EDcl no AgRg no HC n. 653.425/MG, de minha relatoria, Sexta
Turma, DJe 19/11/2021)"""

if globals().get("EXEMPLOS_BUSCA", True):
    mostrar(consonancia(QUERY_EXEMPLO))

### Súmulas

- **[Súmula 676/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=676)**: aderência forte (0.698), vigente. Em razão da Lei n. 13.964/2019, não é mais possível ao juiz, de ofício, decretar ou converter prisão em flagrante em prisão preventiva.
- **[Súmula 409/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=409)**: aderência moderada (0.604), vigente. Em execução fiscal, a prescrição ocorrida antes da propositura da ação pode ser decretada de ofício (art. 219, § 5º, do CPC).
- **[Súmula 668/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=668)**: aderência moderada (0.532), vigente. Não é hediondo o delito de porte ou posse de arma de fogo de uso permitido, ainda que com numeração, marca ou qualquer outro sinal de identificação raspado, suprimido ou adulterado.
- **[Súmula 267/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=267)**: aderência moderada (0.603), vigente. A interposição de recurso, sem efeito suspensivo, contra decisão condenatória não obsta a expedição de mandado de prisão.
- **[Súmula 630/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=630)**: aderência moderada (0.547), alterada. A incidência da atenuante da confissão espontânea no crime de tráfico ilícito de entorpecentes quando o acusado admitir a posse ou a propriedade para uso próprio, negando a prática do tráfico de drogas, deve ocorrer em proporção inferior à que seria devida no caso de confissão plena.

### Precedentes com tese

- **[Tema 1098/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1098&cod_tema_final=1098)**: aderência moderada (0.606), Trânsito em Julgado. 1 - O Acordo de Não Persecução Penal constitui um negócio jurídico processual penal instituído por norma que possui natureza processual, no que diz respeito à possibilidade de composição entre as partes com o fim de evitar a instauração da ação penal, e, de outro lado, natureza material em razão da previsão de extinção da punibilidade de quem cumpre os deveres estabelecidos no acordo (art. 28-A, §
- **[Tema 134/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=134&cod_tema_final=134)**: aderência moderada (0.649), Trânsito em Julgado. Em execução fiscal, a prescrição ocorrida antes da propositura da ação pode ser decretada de ofício (art. 219, § 5º, do CPC).
- **[Tema 1259/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1259&cod_tema_final=1259)**: aderência moderada (0.552), Trânsito em Julgado. A majorante do art. 40, inciso IV, da Lei n. 11.343/2006 aplica-se quando há nexo finalístico entre o uso da arma e o tráfico de drogas, sendo a arma usada para garantir o sucesso da atividade criminosa, hipótese em que o crime de porte ou posse ilegal de arma é absorvido pelo tráfico. Do contrário, o delito previsto no Estatuto do Desarmamento é considerado crime autônomo, em concurso material co
- **[Tema 1347/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1347&cod_tema_final=1347)**: aderência moderada (0.618), Trânsito em Julgado. A regressão cautelar de regime prisional é medida de caráter provisório e está autorizada pelo poder geral de cautela do juízo da execução, podendo ser aplicada, mediantefundamentação idônea, até a apuração definitiva da falta.
- **[Tema 1167/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1167&cod_tema_final=1167)**: aderência moderada (0.610), Trânsito em Julgado. "A audiência prevista no art. 16 da Lei 11.340/2006 tem por objetivo confirmar a retratação, não a representação, e não pode ser designada de ofício pelo juiz. Sua realização somente é necessária caso haja manifestação do desejo da vítima de se retratar trazida aos autos antes do recebimento da denúncia".

### Em andamento (sem tese firmada)

- **[Tema 1431/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1431&cod_tema_final=1431)**: aderência forte (0.666), Em Julgamento. Definir se a solicitação, sem a efetiva entrega do entorpecente ao destinatário no estabelecimento prisional, caracteriza ato preparatório, impunível em razão da atipicidade da conduta, ou se configura conduta típica de tráfico de drogas pela aplicação do art. 29 do Código Penal.
- **[Tema 1407/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1407&cod_tema_final=1407)**: aderência moderada (0.578), Afetado. Definir, em relação à causa de aumento de pena disposta no art. 157, §2º-A, I, do Código Penal, se: 1) é necessária apreensão de arma de fogo; 2) é necessária a perícia da arma de fogo; 3) é necessária tanto a apreensão quanto a perícia; 4) se, na ausência de apreensão e perícia, outros meios probatórios podem ser considerados hábeis para comprovar o uso do artefato.
- **[Tema 1163/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1163&cod_tema_final=1163)**: aderência moderada (0.595), Afetado. Saber se a simples fuga do réu para dentro da residência ao avistar os agentes estatais e/ou a mera existência de denúncia anônima acerca da possível prática de delito no interior do domicílio,desacompanhada de outros elementos preliminares indicativos de crime, constituem ou não, por si sós, fundadas razões (justa causa) a autorizar o ingresso dos policiais em seu domicílio, sem prévia autorizaçã

### Inativos (não usar como fundamento)

- **[Tema 583/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=583&cod_tema_final=583)**: aderência forte (0.698), Cancelado. Questiona-se a possibilidade de concessão de liberdade provisória a preso, em flagrante, pela suposta prática do crime de tráfico ilícito de drogas (art. 33, caput,da Lei n.º 11.343/2006), haja visto a vedação expressa do art. 44 da Lei n.º 11.343/2006.
- **Controvérsia 438/STJ**: aderência forte (0.653), Vinculada a Tema. Estabelecer se, a despeito da guarda municipal não desempenhar a função de policiamento ostensivo, ela pode prender quem esteja em flagrante delito, respaldada no art. 301 do Código de Processo Penal.
